# 00 - Setup: Carga de Dados no SQL Server

Este notebook carrega os arquivos CSV da pasta `data/` para o banco de dados **LojoDB** no SQL Server 2025.

**Schema do banco:** 10 tabelas de um e-commerce fictício cobrindo catálogo de produtos, clientes, pedidos e pagamentos.

**Pré-requisitos:**
- Docker Compose rodando (`docker compose up -d`)
- SQL Server acessível na porta `1433`
- Driver ODBC 18 instalado

## 1. Configuração e Conexão

In [ ]:
import os
import pandas as pd
import pyodbc
from dotenv import load_dotenv

load_dotenv(override=True)

DB_SERVER   = os.getenv('DB_SERVER')
DB_PORT     = os.getenv('DB_PORT')
DB_USER     = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE')

print(f'Servidor: {DB_SERVER}:{DB_PORT}')
print(f'Database: {DB_DATABASE}')

In [ ]:
# Conexão ao SQL Server (master) para criar o database
conn_master = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'UID={DB_USER};'
    f'PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor_master = conn_master.cursor()
print('Conectado ao SQL Server (master) com sucesso!')

## 2. Criar Database LojoDB

In [ ]:
cursor_master.execute(f"""
    IF NOT EXISTS (SELECT name FROM sys.databases WHERE name = '{DB_DATABASE}')
    BEGIN
        CREATE DATABASE [{DB_DATABASE}]
    END
""")
print(f'Database [{DB_DATABASE}] criado/verificado com sucesso!')
cursor_master.close()
conn_master.close()

In [ ]:
conn = pyodbc.connect(
    f'DRIVER={{ODBC Driver 18 for SQL Server}};'
    f'SERVER={DB_SERVER},{DB_PORT};'
    f'DATABASE={DB_DATABASE};'
    f'UID={DB_USER};PWD={DB_PASSWORD};'
    f'TrustServerCertificate=yes;',
    autocommit=True
)
cursor = conn.cursor()
print(f'Conectado ao [{DB_DATABASE}] com sucesso!')

## 3. Criar Tabelas

Modelo de dados do **LojoDB** — e-commerce com 10 tabelas:

```
categoria ──< produto >── produto_fornecedor >── fornecedor
                 │
cliente ──< pedido >── item_pedido >── produto
   │           │
endereco    pagamento

produto ──< avaliacao >── cliente
```

In [ ]:
ddl_statements = [
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'categoria')
    CREATE TABLE categoria (
        cd_categoria  INT          PRIMARY KEY,
        nm_categoria  VARCHAR(100) NOT NULL,
        ds_categoria  VARCHAR(300)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'fornecedor')
    CREATE TABLE fornecedor (
        cd_fornecedor INT          PRIMARY KEY,
        nm_fornecedor VARCHAR(200) NOT NULL,
        cnpj          VARCHAR(20),
        nm_contato    VARCHAR(200),
        email         VARCHAR(200),
        telefone      VARCHAR(30)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'produto')
    CREATE TABLE produto (
        cd_produto    INT           PRIMARY KEY,
        cd_categoria  INT           NOT NULL,
        nm_produto    VARCHAR(200)  NOT NULL,
        ds_produto    VARCHAR(500),
        vl_preco      DECIMAL(10,2) NOT NULL,
        qt_estoque    INT           DEFAULT 0,
        dt_cadastro   DATE
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'produto_fornecedor')
    CREATE TABLE produto_fornecedor (
        cd_produto    INT           NOT NULL,
        cd_fornecedor INT           NOT NULL,
        vl_custo      DECIMAL(10,2),
        dt_contrato   DATE
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'cliente')
    CREATE TABLE cliente (
        cd_cliente    INT          PRIMARY KEY,
        nm_cliente    VARCHAR(200) NOT NULL,
        cpf           VARCHAR(14)  NOT NULL,
        email         VARCHAR(200),
        sexo          CHAR(1),
        dt_nascimento DATE,
        dt_cadastro   DATE
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'endereco')
    CREATE TABLE endereco (
        cd_endereco   INT          PRIMARY KEY,
        cd_cliente    INT          NOT NULL,
        tp_endereco   VARCHAR(30),
        ds_logradouro VARCHAR(300),
        nr_logradouro INT,
        nm_bairro     VARCHAR(200),
        nm_cidade     VARCHAR(200),
        sigla_uf      CHAR(2),
        cd_cep        VARCHAR(10)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pedido')
    CREATE TABLE pedido (
        cd_pedido  INT           PRIMARY KEY,
        cd_cliente INT           NOT NULL,
        dt_pedido  DATE,
        dt_entrega DATE,
        vl_total   DECIMAL(10,2),
        ds_status  VARCHAR(50)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'item_pedido')
    CREATE TABLE item_pedido (
        cd_pedido    INT           NOT NULL,
        cd_produto   INT           NOT NULL,
        qt_item      INT           NOT NULL,
        vl_unitario  DECIMAL(10,2),
        vl_desconto  DECIMAL(10,2) DEFAULT 0
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pagamento')
    CREATE TABLE pagamento (
        cd_pagamento INT           PRIMARY KEY,
        cd_pedido    INT           NOT NULL,
        tp_pagamento VARCHAR(50),
        vl_pagamento DECIMAL(10,2),
        dt_pagamento DATE,
        ds_status    VARCHAR(30)
    )
    """,
    """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'avaliacao')
    CREATE TABLE avaliacao (
        cd_avaliacao  INT          PRIMARY KEY,
        cd_produto    INT          NOT NULL,
        cd_cliente    INT          NOT NULL,
        nr_nota       INT,
        ds_comentario VARCHAR(500),
        dt_avaliacao  DATE
    )
    """
]

for ddl in ddl_statements:
    cursor.execute(ddl)

print('Todas as tabelas foram criadas com sucesso!')

## 4. Carregar Dados dos CSVs

In [ ]:
# Ordem respeita dependências entre tabelas
tabelas = [
    'categoria', 'fornecedor', 'produto', 'produto_fornecedor',
    'cliente', 'endereco', 'pedido', 'item_pedido', 'pagamento', 'avaliacao'
]

# Diretório data/ relativo ao notebook
data_dir = os.path.join(os.path.dirname(os.getcwd()), 'data')
if not os.path.exists(data_dir):
    data_dir = os.path.join(os.getcwd(), 'data')  # fallback se rodando direto

for tabela in tabelas:
    csv_path = os.path.join(data_dir, f'{tabela}.csv')

    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    if count > 0:
        print(f'  ⏭️  {tabela}: já contém {count} registros, pulando...')
        continue

    df = pd.read_csv(csv_path)
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()

    cols = ', '.join(df.columns)
    placeholders = ', '.join(['?' for _ in df.columns])
    insert_sql = f'INSERT INTO {tabela} ({cols}) VALUES ({placeholders})'
    data = [tuple(None if pd.isna(v) else v for v in row) for row in df.itertuples(index=False)]

    for i in range(0, len(data), 2000):
        cursor.executemany(insert_sql, data[i:i+2000])

    print(f'  ✅ {tabela}: {len(data)} registros inseridos')

print('\n🎉 Carga de dados concluída!')

## 5. Validação

In [ ]:
print(f'{"Tabela":<25} {"Registros":>10}')
print('-' * 37)
for tabela in tabelas:
    cursor.execute(f'SELECT COUNT(*) FROM {tabela}')
    count = cursor.fetchone()[0]
    print(f'{tabela:<25} {count:>10}')
print('\n✅ Validação concluída!')

In [ ]:
# Amostra de dados
for tabela in ['categoria', 'produto', 'cliente', 'pedido']:
    print(f'\n--- {tabela.upper()} (primeiros 5 registros) ---')
    cursor.execute(f'SELECT TOP 5 * FROM {tabela}')
    cols = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    import pandas as pd
    df_s = pd.DataFrame.from_records(rows, columns=cols)
    print(df_s.to_string(index=False))

In [ ]:
cursor.close()
conn.close()
print('Conexão encerrada.')